# XML Processing and Cleaning Tutorial for PubMed Central Articles

This tutorial will guide you through understanding, parsing, and cleaning XML files from PubMed Central (PMC) for use in machine learning and NLP applications.

---

## Table of Contents

1. [Understanding XML Structure](#1-understanding-xml-structure)
2. [Python Libraries for XML Processing](#2-python-libraries-for-xml-processing)
3. [Parsing XML Files](#3-parsing-xml-files)
4. [Extracting Specific Content](#4-extracting-specific-content)
5. [Cleaning and Preprocessing Text](#5-cleaning-and-preprocessing-text)
6. [Complete Pipeline Example](#6-complete-pipeline-example)
7. [Batch Processing Multiple Files](#7-batch-processing-multiple-files)
8. [Common Issues and Solutions](#8-common-issues-and-solutions)

---

## 1. Understanding XML Structure

### What is XML?

XML (eXtensible Markup Language) is a hierarchical format that stores data in nested tags:

```xml
<parent>
    <child attribute="value">Content</child>
</parent>
```

### PMC XML Structure

PMC articles follow the JATS (Journal Article Tag Suite) XML format:

```
<pmc-articleset>
  └── <article>
      ├── <front>          # Metadata (title, authors, journal, etc.)
      ├── <body>           # Main article content
      └── <back>           # References, acknowledgments
```

### Key Components

| Section | Contains | Use Case |
|---------|----------|----------|
| `<front>` | Title, authors, abstract, keywords | Metadata extraction |
| `<body>` | Introduction, methods, results, discussion | Main text for training |
| `<back>` | References, acknowledgments | Citation analysis |

---

## 2. Python Libraries for XML Processing

### Installation

```bash
pip install lxml beautifulsoup4
```

### Library Comparison

| Library | Pros | Cons | Best For |
|---------|------|------|----------|
| **lxml** | Fast, XPath support | Complex API | Large files, precise extraction |
| **xml.etree.ElementTree** | Built-in, simple | Slower | Small files, basic parsing |
| **BeautifulSoup** | Easy, forgiving | Slower | Malformed XML, quick prototyping |

---

## 3. Parsing XML Files

### Method 1: Using lxml (Recommended)

```python
from lxml import etree

def parse_xml_file(filepath):
    """Parse XML file and return root element."""
    try:
        tree = etree.parse(filepath)
        root = tree.getroot()
        return root
    except etree.XMLSyntaxError as e:
        print(f"XML parsing error: {e}")
        return None

# Usage
root = parse_xml_file('PMC6143471.xml')
print(f"Root tag: {root.tag}")
```

### Method 2: Using ElementTree

```python
import xml.etree.ElementTree as ET

def parse_xml_elementtree(filepath):
    """Parse XML using ElementTree."""
    tree = ET.parse(filepath)
    root = tree.getroot()
    return root

# Usage
root = parse_xml_elementtree('PMC6143471.xml')
```

### Method 3: Using BeautifulSoup

```python
from bs4 import BeautifulSoup

def parse_xml_bs4(filepath):
    """Parse XML using BeautifulSoup."""
    with open(filepath, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'lxml-xml')
    return soup

# Usage
soup = parse_xml_bs4('PMC6143471.xml')
```

---

## 4. Extracting Specific Content

### Extract Article Title

```python
def extract_title(root):
    """Extract article title from XML."""
    # Using lxml with namespace handling
    title = root.find('.//article-title')
    if title is not None:
        return title.text
    return None

# Example
title = extract_title(root)
print(f"Title: {title}")
```

### Extract Abstract

```python
def extract_abstract(root):
    """Extract abstract sections."""
    abstract_parts = []
    
    # Find all abstract sections
    abstract = root.find('.//abstract')
    if abstract is not None:
        for sec in abstract.findall('.//sec'):
            # Get section title
            title_elem = sec.find('.//title')
            title = title_elem.text if title_elem is not None else ""
            
            # Get section paragraphs
            paragraphs = []
            for p in sec.findall('.//p'):
                # Get all text including nested elements
                text = ''.join(p.itertext())
                paragraphs.append(text.strip())
            
            if title or paragraphs:
                abstract_parts.append({
                    'title': title,
                    'content': ' '.join(paragraphs)
                })
    
    return abstract_parts

# Example
abstract = extract_abstract(root)
for section in abstract:
    print(f"\n{section['title']}:")
    print(section['content'][:200] + "...")
```

### Extract Full Body Text

```python
def extract_body_text(root):
    """Extract all body text from article."""
    body_text = []
    
    body = root.find('.//body')
    if body is None:
        return ""
    
    # Iterate through all sections
    for sec in body.findall('.//sec'):
        # Get section title
        title_elem = sec.find('./title')
        if title_elem is not None:
            body_text.append(f"\n## {title_elem.text}\n")
        
        # Get all paragraphs in this section
        for p in sec.findall('.//p'):
            # Extract text from paragraph and all nested elements
            para_text = ''.join(p.itertext()).strip()
            if para_text:
                body_text.append(para_text)
    
    return '\n\n'.join(body_text)

# Example
body = extract_body_text(root)
print(body[:500])
```

### Extract Authors

```python
def extract_authors(root):
    """Extract author information."""
    authors = []
    
    for contrib in root.findall('.//contrib[@contrib-type="author"]'):
        name_elem = contrib.find('.//name')
        if name_elem is not None:
            surname = name_elem.find('surname')
            given_names = name_elem.find('given-names')
            
            author = {
                'surname': surname.text if surname is not None else "",
                'given_names': given_names.text if given_names is not None else "",
                'full_name': f"{given_names.text if given_names is not None else ''} {surname.text if surname is not None else ''}".strip()
            }
            authors.append(author)
    
    return authors

# Example
authors = extract_authors(root)
for author in authors:
    print(author['full_name'])
```

### Extract Keywords

```python
def extract_keywords(root):
    """Extract article keywords."""
    keywords = []
    
    kwd_group = root.find('.//kwd-group')
    if kwd_group is not None:
        for kwd in kwd_group.findall('.//kwd'):
            if kwd.text:
                keywords.append(kwd.text.strip())
    
    return keywords

# Example
keywords = extract_keywords(root)
print(f"Keywords: {', '.join(keywords)}")
```

---

## 5. Cleaning and Preprocessing Text

### Remove Special Characters

```python
import re

def clean_text(text):
    """Clean text by removing special characters and normalizing whitespace."""
    if not text:
        return ""
    
    # Remove XML entities
    text = text.replace('&lt;', '<').replace('&gt;', '>').replace('&amp;', '&')
    
    # Remove special Unicode characters
    text = re.sub(r'[\u2018\u2019]', "'", text)  # Smart quotes to regular quotes
    text = re.sub(r'[\u201c\u201d]', '"', text)  # Smart double quotes
    text = re.sub(r'\u2013|\u2014', '-', text)   # En/em dashes to hyphens
    
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    return text

# Example
dirty_text = "This is  a   test's text with—special chars"
clean = clean_text(dirty_text)
print(clean)
```

### Remove Citations and References

```python
def remove_citations(text):
    """Remove citation markers like [1], [2,3], etc."""
    # Remove square bracket citations
    text = re.sub(r'\[\d+(?:,\s*\d+)*\]', '', text)
    
    # Remove parenthetical citations like (Author, 2020)
    text = re.sub(r'\([A-Z][a-z]+(?:\s+et\s+al\.)?,\s+\d{4}\)', '', text)
    
    # Clean up extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Example
text_with_citations = "This is a fact [1,2,3]. Another fact (Smith et al., 2020)."
clean = remove_citations(text_with_citations)
print(clean)
```

### Remove Tables and Figures References

```python
def remove_table_figure_refs(text):
    """Remove references to tables and figures."""
    # Remove table references
    text = re.sub(r'\((?:Table|Tab\.)\s+\d+\)', '', text, flags=re.IGNORECASE)
    
    # Remove figure references
    text = re.sub(r'\((?:Figure|Fig\.)\s+\d+[A-Z]?\)', '', text, flags=re.IGNORECASE)
    
    # Clean up spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Example
text = "As shown in (Table 1) and (Figure 2A), the results are significant."
clean = remove_table_figure_refs(text)
print(clean)
```

### Remove Italic/Bold Markup

```python
def extract_text_from_element(element):
    """Extract plain text from XML element, removing all markup."""
    if element is None:
        return ""
    
    # Get all text content, ignoring tags
    text = ''.join(element.itertext())
    
    return clean_text(text)

# Example with lxml element
para = root.find('.//p')
plain_text = extract_text_from_element(para)
print(plain_text)
```

### Complete Text Cleaning Pipeline

```python
def clean_extracted_text(text):
    """Apply all cleaning steps to extracted text."""
    if not text:
        return ""
    
    # Step 1: Clean special characters
    text = clean_text(text)
    
    # Step 2: Remove citations
    text = remove_citations(text)
    
    # Step 3: Remove table/figure references
    text = remove_table_figure_refs(text)
    
    # Step 4: Remove extra punctuation
    text = re.sub(r'[;]{2,}', ';', text)  # Multiple semicolons
    text = re.sub(r'\.{2,}', '.', text)   # Multiple periods
    
    # Step 5: Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Example
dirty = "This is  a test [1,2]. See (Table 1) for details—more text."
clean = clean_extracted_text(dirty)
print(clean)
```

---

## 6. Complete Pipeline Example

### Full Article Extraction and Cleaning

```python
from lxml import etree
import re

class PMCArticleExtractor:
    """Extract and clean content from PMC XML files."""
    
    def __init__(self, xml_path):
        """Initialize with path to XML file."""
        self.xml_path = xml_path
        self.root = self._parse_xml()
    
    def _parse_xml(self):
        """Parse XML file."""
        try:
            tree = etree.parse(self.xml_path)
            return tree.getroot()
        except Exception as e:
            print(f"Error parsing {self.xml_path}: {e}")
            return None
    
    def extract_metadata(self):
        """Extract article metadata."""
        if self.root is None:
            return {}
        
        metadata = {
            'pmid': self._get_text('.//article-id[@pub-id-type="pmid"]'),
            'pmcid': self._get_text('.//article-id[@pub-id-type="pmcid"]'),
            'doi': self._get_text('.//article-id[@pub-id-type="doi"]'),
            'title': self._get_text('.//article-title'),
            'journal': self._get_text('.//journal-title'),
            'year': self._get_text('.//pub-date[@pub-type="ppub"]/year'),
            'authors': self._extract_authors(),
            'keywords': self._extract_keywords()
        }
        
        return metadata
    
    def _get_text(self, xpath):
        """Get text from XPath query."""
        elem = self.root.find(xpath)
        if elem is not None:
            return ''.join(elem.itertext()).strip()
        return None
    
    def _extract_authors(self):
        """Extract author names."""
        authors = []
        for contrib in self.root.findall('.//contrib[@contrib-type="author"]'):
            name = contrib.find('.//name')
            if name is not None:
                surname = name.find('surname')
                given = name.find('given-names')
                full_name = f"{given.text if given is not None else ''} {surname.text if surname is not None else ''}".strip()
                if full_name:
                    authors.append(full_name)
        return authors
    
    def _extract_keywords(self):
        """Extract keywords."""
        keywords = []
        for kwd in self.root.findall('.//kwd'):
            if kwd.text:
                keywords.append(kwd.text.strip())
        return keywords
    
    def extract_abstract(self):
        """Extract and clean abstract."""
        abstract_parts = []
        
        abstract = self.root.find('.//abstract')
        if abstract is not None:
            # Get all paragraphs
            for p in abstract.findall('.//p'):
                text = ''.join(p.itertext()).strip()
                if text:
                    abstract_parts.append(text)
        
        full_abstract = ' '.join(abstract_parts)
        return self._clean_text(full_abstract)
    
    def extract_body(self):
        """Extract and clean body text."""
        body_parts = []
        
        body = self.root.find('.//body')
        if body is not None:
            for sec in body.findall('.//sec'):
                # Get section title
                title = sec.find('./title')
                if title is not None and title.text:
                    body_parts.append(f"\n{title.text}\n")
                
                # Get paragraphs
                for p in sec.findall('.//p'):
                    text = ''.join(p.itertext()).strip()
                    if text:
                        body_parts.append(text)
        
        full_body = '\n\n'.join(body_parts)
        return self._clean_text(full_body)
    
    def extract_full_text(self):
        """Extract complete article text (abstract + body)."""
        abstract = self.extract_abstract()
        body = self.extract_body()
        
        parts = []
        if abstract:
            parts.append(f"ABSTRACT\n\n{abstract}")
        if body:
            parts.append(f"MAIN TEXT\n\n{body}")
        
        return '\n\n'.join(parts)
    
    def _clean_text(self, text):
        """Clean extracted text."""
        if not text:
            return ""
        
        # Remove citations
        text = re.sub(r'\[\d+(?:,\s*\d+)*\]', '', text)
        text = re.sub(r'\([A-Z][a-z]+(?:\s+et\s+al\.)?,\s+\d{4}\)', '', text)
        
        # Remove table/figure references
        text = re.sub(r'\((?:Table|Tab\.|Figure|Fig\.)\s+\d+[A-Z]?\)', '', text, flags=re.IGNORECASE)
        
        # Clean special characters
        text = text.replace('&lt;', '<').replace('&gt;', '>').replace('&amp;', '&')
        text = re.sub(r'[\u2018\u2019]', "'", text)
        text = re.sub(r'[\u201c\u201d]', '"', text)
        text = re.sub(r'\u2013|\u2014', '-', text)
        
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\n\s*\n', '\n\n', text)
        
        return text.strip()
    
    def to_dict(self):
        """Convert article to dictionary."""
        return {
            'metadata': self.extract_metadata(),
            'abstract': self.extract_abstract(),
            'body': self.extract_body(),
            'full_text': self.extract_full_text()
        }

# Usage Example
extractor = PMCArticleExtractor('PMC6143471.xml')

# Get metadata
metadata = extractor.extract_metadata()
print(f"Title: {metadata['title']}")
print(f"Authors: {', '.join(metadata['authors'][:3])}")
print(f"PMID: {metadata['pmid']}")

# Get abstract
abstract = extractor.extract_abstract()
print(f"\nAbstract ({len(abstract)} chars):")
print(abstract[:300] + "...")

# Get full text
full_text = extractor.extract_full_text()
print(f"\nFull text length: {len(full_text)} characters")

# Save to file
with open('cleaned_article.txt', 'w', encoding='utf-8') as f:
    f.write(full_text)
```

---

## 7. Batch Processing Multiple Files

### Process Directory of XML Files

```python
import os
import json
from pathlib import Path

def process_xml_directory(input_dir, output_dir):
    """Process all XML files in a directory."""
    
    # Create output directory
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Get all XML files
    xml_files = list(Path(input_dir).glob('*.xml'))
    print(f"Found {len(xml_files)} XML files")
    
    results = []
    errors = []
    
    for i, xml_file in enumerate(xml_files, 1):
        print(f"Processing {i}/{len(xml_files)}: {xml_file.name}")
        
        try:
            # Extract content
            extractor = PMCArticleExtractor(str(xml_file))
            article_data = extractor.to_dict()
            
            # Save cleaned text
            pmcid = article_data['metadata'].get('pmcid', xml_file.stem)
            output_file = Path(output_dir) / f"{pmcid}.txt"
            
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(article_data['full_text'])
            
            # Save metadata
            metadata_file = Path(output_dir) / f"{pmcid}_metadata.json"
            with open(metadata_file, 'w', encoding='utf-8') as f:
                json.dump(article_data['metadata'], f, indent=2)
            
            results.append({
                'file': xml_file.name,
                'pmcid': pmcid,
                'status': 'success',
                'text_length': len(article_data['full_text'])
            })
            
        except Exception as e:
            print(f"  Error: {e}")
            errors.append({
                'file': xml_file.name,
                'error': str(e)
            })
    
    # Save processing report
    report = {
        'total_files': len(xml_files),
        'successful': len(results),
        'failed': len(errors),
        'results': results,
        'errors': errors
    }
    
    with open(Path(output_dir) / 'processing_report.json', 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"\nProcessing complete:")
    print(f"  Successful: {len(results)}")
    print(f"  Failed: {len(errors)}")
    
    return report

# Usage
report = process_xml_directory(
    input_dir='../fetched_articles/fetched_pmc_xmls/2020',
    output_dir='../cleaned_articles/2020'
)
```

### Create Training Dataset

```python
def create_training_dataset(cleaned_dir, output_file):
    """Combine cleaned articles into a single training file."""
    
    all_text = []
    
    for txt_file in Path(cleaned_dir).glob('*.txt'):
        if txt_file.name == 'training_data.txt':
            continue
        
        with open(txt_file, 'r', encoding='utf-8') as f:
            text = f.read().strip()
            if text:
                all_text.append(text)
                all_text.append('\n\n' + '='*80 + '\n\n')  # Separator
    
    # Write combined file
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(''.join(all_text))
    
    print(f"Created training dataset: {output_file}")
    print(f"  Total articles: {len(all_text) // 2}")
    print(f"  Total size: {len(''.join(all_text)):,} characters")

# Usage
create_training_dataset(
    cleaned_dir='../cleaned_articles/2020',
    output_file='../training_data/liver_transplant_2020.txt'
)
```

---

## 8. Common Issues and Solutions

### Issue 1: Namespace Errors

**Problem**: XML uses namespaces that prevent finding elements

```python
# BAD - Won't work with namespaces
title = root.find('article-title')

# GOOD - Use .// to search anywhere
title = root.find('.//article-title')

# BETTER - Handle namespaces explicitly
namespaces = {'ns': 'http://www.example.com/namespace'}
title = root.find('.//ns:article-title', namespaces)
```

### Issue 2: Missing Elements

**Problem**: Not all articles have all elements

```python
# BAD - Will crash if element doesn't exist
title = root.find('.//article-title').text

# GOOD - Check if element exists
title_elem = root.find('.//article-title')
title = title_elem.text if title_elem is not None else "No title"

# BETTER - Use helper function
def safe_get_text(root, xpath, default=""):
    elem = root.find(xpath)
    return elem.text if elem is not None else default
```

### Issue 3: Nested Text Elements

**Problem**: Text split across multiple nested tags

```python
# BAD - Only gets direct text
text = paragraph.text

# GOOD - Get all text including nested elements
text = ''.join(paragraph.itertext())
```

### Issue 4: Memory Issues with Large Files

**Problem**: Loading huge XML files into memory

```python
# BAD - Loads entire file
tree = etree.parse('huge_file.xml')

# GOOD - Use iterparse for streaming
def stream_parse_xml(filepath):
    """Parse large XML files efficiently."""
    for event, elem in etree.iterparse(filepath, events=('end',), tag='article'):
        # Process element
        yield process_article(elem)
        
        # Clear element to free memory
        elem.clear()
        while elem.getprevious() is not None:
            del elem.getparent()[0]
```

### Issue 5: Encoding Problems

**Problem**: Special characters cause errors

```python
# GOOD - Always specify encoding
with open(xml_file, 'r', encoding='utf-8') as f:
    content = f.read()

# BETTER - Handle encoding errors
with open(xml_file, 'r', encoding='utf-8', errors='ignore') as f:
    content = f.read()
```

---

## Quick Reference: Common XPath Queries

```python
# Article metadata
'.//article-id[@pub-id-type="pmid"]'      # PMID
'.//article-id[@pub-id-type="pmcid"]'     # PMCID
'.//article-id[@pub-id-type="doi"]'       # DOI
'.//article-title'                         # Title
'.//journal-title'                         # Journal name

# Authors and affiliations
'.//contrib[@contrib-type="author"]'       # All authors
'.//contrib//name/surname'                 # Author surnames
'.//contrib//name/given-names'             # Author given names
'.//aff'                                   # Affiliations

# Content
'.//abstract'                              # Abstract
'.//abstract//p'                           # Abstract paragraphs
'.//body'                                  # Body
'.//body//sec'                             # Body sections
'.//body//p'                               # Body paragraphs

# Keywords and subjects
'.//kwd-group/kwd'                         # Keywords
'.//subject'                               # Subject categories

# References
'.//ref-list/ref'                          # All references
'.//ref//article-title'                    # Reference titles
```

---

## Summary

### Best Practices

1. **Always use `try-except`** when parsing XML files
2. **Check if elements exist** before accessing their properties
3. **Use `.itertext()`** to get all nested text
4. **Clean text incrementally** (citations → figures → special chars)
5. **Save metadata separately** from full text
6. **Process in batches** for large datasets
7. **Log errors** for debugging

### Recommended Workflow

```
1. Parse XML → 2. Extract metadata → 3. Extract content → 
4. Clean text → 5. Save outputs → 6. Generate report
```

This tutorial provides everything you need to process PMC XML files for machine learning applications!
